# scBaseCount Homo sapiens metadata (parquet)

This notebook works through the questions in `DailyNotes.md` (2026-03-23):

- What do the parquet columns represent, and how do `obs_metadata` vs `sample_metadata` relate?
- At Homo sapiens scale: how many SRX accessions (studies / processed experiments), and how many cells?
- For **cancer**-related records: which diseases appear, and how many distinct labels (subcategories)?
- For **immune-oncology (IO)**-relevant records: which SRX rows look IO-related via `perturbation` / `disease` / `tissue`, and how many cells and subcategories per slice?
- What is available about **patient-level** fields (e.g. cancer stage) vs what is **not** in these tables?
- **IO × disease-theme cohort cards:** the same workflow repeated per disease area (e.g. breast, lung, melanoma)—**cells + SRX counts only**, no patient rollups—plus top `disease` / `perturbation` lines for a narrative card.

**Data layout:** point `DATA_ROOT` at a scBasecount drop that contains `metadata/GeneFull/Homo_sapiens/*.parquet` (or adjust the glob below).

In [128]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Default: repository layout under data/ (gitignored). Override if your parquet lives elsewhere.
REPO_ROOT = Path("..").resolve()
DATA_ROOT = REPO_ROOT / "data" / "scbasecount" / "2026-01-12"

META_GLOB = "**/metadata/**/Homo_sapiens/*Homo_sapiens*_metadata*.parquet"

paths = sorted(DATA_ROOT.glob(META_GLOB))
if not paths:
    raise FileNotFoundError(
        f"No Homo sapiens parquet under {DATA_ROOT!s}. Set DATA_ROOT to your scBaseCount metadata folder."
    )

obs_paths = [p for p in paths if "obs_metadata" in p.name]
sample_paths = [p for p in paths if "sample_metadata" in p.name]
assert len(obs_paths) == 1 and len(sample_paths) == 1, (obs_paths, sample_paths)
OBS_PARQUET = obs_paths[0]
SAMPLE_PARQUET = sample_paths[0]
# OBS_PARQUET, SAMPLE_PARQUET

## 1. Column reference (what each file stores)

| File | Grain | Role |
|------|--------|------|
| `*_obs_metadata.parquet` | one row per **cell** | Barcodes, SRX accession, QC-ish gene/UMI counts, optional `cell_type` / `cell_ontology_term_id`. |
| `*_sample_metadata.parquet` | one row per **SRX** (processed experiment accession) | Study-level annotations Arc attached: tissue, disease, perturbation, `obs_count` (cells in that SRX), paths to `h5ad`, etc. |

Patient identifiers (e.g. individual donor counts, clinical stage) are **not** guaranteed in these columns; see the gap analysis section below.

In [129]:
obs_schema = pq.read_schema(OBS_PARQUET)
sample_schema = pq.read_schema(SAMPLE_PARQUET)

pd.DataFrame(
    {"obs_metadata": obs_schema.names, "dtype": [str(obs_schema.field(n).type) for n in obs_schema.names]}
)

,obs_metadata,dtype
0,cell_barcode,string
1,SRX_accession,string
2,gene_count_Unique,int64
3,umi_count_Unique,float
4,gene_count_UniqueAndMult-EM,int64
5,umi_count_UniqueAndMult-EM,float
6,gene_count_UniqueAndMult-Uniform,int64
7,umi_count_UniqueAndMult-Uniform,float
8,cell_type,string
9,cell_ontology_term_id,string


In [130]:
pd.DataFrame(
    {
        "sample_metadata": sample_schema.names,
        "dtype": [str(sample_schema.field(n).type) for n in sample_schema.names],
    }
)

,sample_metadata,dtype
0,entrez_id,int64
1,srx_accession,string
2,file_path,string
3,obs_count,int64
4,lib_prep,string
5,tech_10x,string
6,cell_prep,string
7,organism,string
8,tissue,string
9,tissue_ontology_term_id,string


## 2. Homo sapiens scale: SRX rows and cells

Use `sample_metadata.obs_count` summed for total cells (matches one row per cell in `obs_metadata` if the pipeline is consistent). Cross-check with parquet row count on `obs_metadata`.

In [ ]:
# Number of cells = number of rows in obs_metadata
pf_obs = pq.ParquetFile(OBS_PARQUET)
n_cells_obs_file = pf_obs.metadata.num_rows
n_cells_obs_file

302277013

In [ ]:
# Number of SRX = number of rows in sample_metadata
# Each SRX has a number of cells = obs_count
sample = pd.read_parquet(SAMPLE_PARQUET)
sample["obs_count"] = pd.to_numeric(sample["obs_count"], errors="coerce")

n_srx = sample["srx_accession"].nunique()
n_cells_from_sample = int(sample["obs_count"].sum())
print(f"SRX rows (sample_metadata): {len(sample):,}")
print(f"Unique srx_accession: {n_srx:,}")
print(f"Sum of obs_count (cells, from sample table): {n_cells_from_sample:,}")
print(f"obs_metadata parquet row count: {n_cells_obs_file:,}")

SRX rows (sample_metadata): 35,266
Unique srx_accession: 35,266
Sum of obs_count (cells, from sample table): 302,277,013
obs_metadata parquet row count: 302,277,013


### Disease and tissue labels (exploratory)

Summaries below use the **full** Homo sapiens `sample_metadata` table (every SRX). Counts are **weighted by `obs_count`** (cells) and include **how many SRX rows** carry each label. Use this to sanity-check regex **themes** in section 5b.

In [133]:
def summarize_labels(
    df: pd.DataFrame, col: str, weight: str = "obs_count", top_n: int = 40
) -> pd.DataFrame:
    w = pd.to_numeric(df[weight], errors="coerce").fillna(0)
    tmp = df.assign(_w=w)
    g = tmp.groupby(col, dropna=False)._w.agg(["sum", "count"])
    g = g.rename(columns={"sum": "cells", "count": "n_srx_rows"}).astype({"cells": "int64"})
    return g.sort_values("cells", ascending=False).head(top_n)


print(
    f"disease: {sample['disease'].isna().sum():,} nulls, "
    f"{sample['disease'].nunique():,} distinct strings"
)
print(
    f"tissue:  {sample['tissue'].isna().sum():,} nulls, "
    f"{sample['tissue'].nunique():,} distinct strings"
)

top_disease = summarize_labels(sample, "disease")
top_tissue = summarize_labels(sample, "tissue")

from IPython.display import display

display(top_disease)
display(top_tissue)

disease: 472 nulls, 6,231 distinct strings
tissue:  0 nulls, 7,247 distinct strings


,cells,n_srx_rows
disease,,
unsure,88006656,8224
chronic myelogenous leukemia,13421381,935
none,11161092,1319
not specified,7360937,974
acute T cell leukemia,5550486,513
Chronic Myelogenous Leukemia,4336204,298
none reported,4146231,557
COVID-19,3966630,457
other,2886005,284


,cells,n_srx_rows
tissue,,
unsure,48500164,4052
CML,19050528,1342
K562 cell line,13318241,923
blood,10546043,1630
acute T cell leukemia,5361704,487
lung,4496488,668
bone marrow,4479187,655
Peripheral Blood Mononuclear Cells (PBMC),4374740,462
Peripheral blood mononuclear cells (PBMCs),3942361,372


## 3. Cancer-related slice: diseases and subcategory counts

"Subcategories" here means distinct `disease` strings after a broad cancer keyword filter (no ontology merge—raw labels as in the table).

In [134]:
CANCER_RE = re.compile(
    r"cancer|carcinoma|melanoma|lymphoma|leukemia|leukaemia|myeloma|sarcoma|tumor|tumour|neoplasm|oma\b|glioma|blastoma",
    re.I,
)


def matches_cancer(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.contains(CANCER_RE, na=False)


cancer_mask = matches_cancer(sample["disease"])
cancer = sample.loc[cancer_mask].copy()
print(
    f"Cancer-like disease label: {len(cancer):,} SRX rows, "
    f"{cancer['disease'].nunique():,} distinct disease strings, "
    f"{int(cancer['obs_count'].sum()):,} cells (obs_count sum)"
)
cancer["disease"].value_counts().head(40)

Cancer-like disease label: 9,596 SRX rows, 1,694 distinct disease strings, 82,373,286 cells (obs_count sum)


disease
chronic myelogenous leukemia                     935
acute T cell leukemia                            513
Chronic Myelogenous Leukemia                     298
breast cancer                                    205
glioblastoma                                     190
multiple myeloma                                 186
melanoma                                         157
Chronic Myelogenous Leukemia (CML)               136
Acute T cell leukemia                            133
colorectal cancer                                124
lung adenocarcinoma                              121
acute myeloid leukemia (AML)                     115
high-grade serous ovarian cancer                 108
hepatocellular carcinoma                          92
Hepatocellular carcinoma                          86
Head and neck squamous cell carcinoma (HNSCC)     79
chronic myelogenous leukemia (CML)                79
prostate cancer                                   78
endometrial carcinoma                 

## 4. Immune-oncology (IO) candidate SRX rows

Heuristic: flag rows where **any** of `perturbation`, `disease`, or `tissue` matches IO / checkpoint / CAR-T style language. This is intentionally broad for **dataset targeting**; tighten keywords for stricter cohorts.

In [135]:
IO_RE = re.compile(
    r"immuno|checkpoint|PD-1|PD1|PD-L1|PDL1|pembrolizumab|nivolumab|atezolizumab|"
    r"durvalumab|ipilimumab|CTLA-4|CTLA4|CAR[- ]?T|CART|anti-PD|ICB|KEYTRUDA|OPDIVO|"
    r"immune checkpoint|checkpoint inhibitor",
    re.I,
)


def io_mask_series(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.contains(IO_RE, na=False)


io_any = (
    io_mask_series(sample["perturbation"])
    | io_mask_series(sample["disease"])
    | io_mask_series(sample["tissue"])
)
io = sample.loc[io_any].copy()
print(
    f"IO keyword hit (any column): {len(io):,} SRX rows, "
    f"{int(io['obs_count'].sum()):,} cells"
)
print("By column (rows can overlap):")
for col in ["perturbation", "disease", "tissue"]:
    m = io_mask_series(sample[col])
    sub = sample.loc[m]
    print(
        f"  {col}: {len(sub):,} SRX, {int(sub['obs_count'].sum()):,} cells, "
        f"{sub[col].nunique():,} distinct {col} strings"
    )

IO keyword hit (any column): 1,853 SRX rows, 13,439,661 cells
By column (rows can overlap):
  perturbation: 1,706 SRX, 12,356,190 cells, 820 distinct perturbation strings
  disease: 211 SRX, 1,554,175 cells, 83 distinct disease strings
  tissue: 105 SRX, 1,072,643 cells, 60 distinct tissue strings


In [136]:
io["io_via"] = ""
io.loc[io_mask_series(io["perturbation"]), "io_via"] += "perturbation;"
io.loc[io_mask_series(io["disease"]), "io_via"] += "disease;"
io.loc[io_mask_series(io["tissue"]), "io_via"] += "tissue;"

card_cols = [
    "srx_accession",
    "disease",
    "perturbation",
    "tissue",
    "cell_line",
    "lib_prep",
    "obs_count",
    "io_via",
    "file_path",
]
card_cols = [c for c in card_cols if c in io.columns]
io.sort_values("obs_count", ascending=False).head(25)[card_cols]

,srx_accession,disease,perturbation,tissue,cell_line,lib_prep,obs_count,io_via,file_path
6117,SRX15016115,bone marrow failure after CD19 CAR-T for Richt...,CD19-directed CAR T-cell therapy (tisagenlecle...,peripheral blood,other,10x_Genomics,146884,perturbation;disease;,gs://arc-institute-virtual-cell-atlas/scbaseco...
13786,SRX22671373,disease stage IV,Nivolumab (Nivo),blood and tumor,live CD45+ immune cells,10x_Genomics,118050,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
3269,SRX22671374,advanced melanoma,combination therapy with relatimab and nivolumab,tumor tissue and blood,not applicable,10x_Genomics,114156,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
26162,SRX15016116,bone marrow failure and DLBCL,CD19 CAR-T therapy (tisagenlecleucel),peripheral blood,not specified; sample derived from a patient,10x_Genomics,102711,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
13517,SRX23973422,acute lymphocytic leukemia,"CAR T therapy, IL-4 addition",PBMCs,CART_ADT_T,10x_Genomics,102039,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
21638,SRX10414128,muscle-invasive bladder cancer,treatment-naïve (no prior neoadjuvant chemothe...,muscle-invasive bladder cancer tumor,not_applicable,10x_Genomics,65242,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
28294,SRX27733857,Head and neck squamous cell carcinoma (HNSCC),"Immunotherapy regimens including anti-PD-1, an...",Tumor tissue from head and neck,No established cell line; primary immune cells...,10x_Genomics,55277,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
14812,SRX26487601,non-small cell lung cancer,nivolumab+chemotherapy,adjacent normal lung,unsure,10x_Genomics,52229,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
27791,SRX27409352,checkpoint inhibitor-induced liver injury (ChILI),Ipilimumab + Nivolumab treatment,peripheral blood mononuclear cells (PBMCs),none,10x_Genomics,45806,perturbation;disease;,gs://arc-institute-virtual-cell-atlas/scbaseco...
33836,SRX28141444,none specified,ethanol,fecal immunochemical test (FIT) sample,none specified,10x_Genomics,40125,tissue;,gs://arc-institute-virtual-cell-atlas/scbaseco...


### IO ∩ cancer (prioritize oncology-relevant IO cohorts)

In [137]:
io_cancer = io.loc[matches_cancer(io["disease"]) | matches_cancer(io["tissue"])].copy()
print(
    f"IO + cancer-like disease or tissue: {len(io_cancer):,} SRX, "
    f"{int(io_cancer['obs_count'].sum()):,} cells"
)
io_cancer.groupby("disease", dropna=False)["obs_count"].sum().sort_values(ascending=False).head(25)

IO + cancer-like disease or tissue: 1,412 SRX, 9,911,983 cells


disease
multiple myeloma                                 1154158
Head and neck squamous cell carcinoma (HNSCC)     427893
melanoma                                          426627
Multiple Myeloma, early relapse                   420943
multiple myeloma, early relapse                   397065
endometrial carcinoma                             392003
Multiple Myeloma                                  329530
advanced melanoma                                 249631
Head and neck squamous cell carcinoma             241144
head and neck squamous cell carcinoma             207437
cancer                                            175636
hepatocellular carcinoma                          174340
cancer immunotherapy                              145271
follicular lymphoma                               144690
Multiple Myeloma, Early relapse                   140020
Neuroblastoma                                     130763
disease stage IV                                  123178
non-small cell lung can

### IO cohorts: cells and SRX counts per disease / collection

In this metadata, each **SRX accession** is one processed experiment (Arc’s unit linking `h5ad` and `obs_metadata` rows). It is **not** the same as individual human donors; patient *n* usually requires GEO/SRA sample attributes outside this parquet.

In [138]:
def cohort_summary(df: pd.DataFrame, key: str) -> pd.DataFrame:
    g = df.groupby(key, dropna=False).agg(
        n_srx=("srx_accession", "nunique"),
        n_cells=("obs_count", "sum"),
    )
    return g.sort_values("n_cells", ascending=False)


cohort_summary(io, "disease").head(20)

,n_srx,n_cells
disease,,
multiple myeloma,119,1154158
unsure,57,481189
Head and neck squamous cell carcinoma (HNSCC),70,427893
melanoma,59,426627
"Multiple Myeloma, early relapse",40,420943
"multiple myeloma, early relapse",45,397065
endometrial carcinoma,61,392003
Multiple Myeloma,36,329530
Checkpoint inhibitor-induced liver injury (ChILI),45,287952


In [139]:
sub = io.dropna(subset=["czi_collection_name"])
if len(sub):
    cohort_summary(sub, "czi_collection_name").head(20)
else:
    print("czi_collection_name all null for IO slice in this snapshot.")

## 5. Per-SRX "data card" helper (disease, treatment modality, tissue, size)

Use `format_study_card` for **one SRX row**. For **cohort-level** IO summaries by disease area (many SRX, cells + labels), use **section 5b** below.

In [140]:
def format_study_card(row: pd.Series) -> str:
    return (
        f"SRX: {row.get('srx_accession', '')}\n"
        f"Disease: {row.get('disease', '')}\n"
        f"Tissue: {row.get('tissue', '')}\n"
        f"Treatment / perturbation: {row.get('perturbation', '')}\n"
        f"Cell line: {row.get('cell_line', '')}\n"
        f"Library prep: {row.get('lib_prep', '')}\n"
        f"Cells (obs_count): {int(row.get('obs_count', 0) or 0):,}\n"
        f"h5ad path: {row.get('file_path', '')}"
    )


example = io.sort_values("obs_count", ascending=False).iloc[0]
print(format_study_card(example))

SRX: SRX15016115
Disease: bone marrow failure after CD19 CAR-T for Richter transformed DLBCL
Tissue: peripheral blood
Treatment / perturbation: CD19-directed CAR T-cell therapy (tisagenlecleucel)
Cell line: other
Library prep: 10x_Genomics
Cells (obs_count): 146,884
h5ad path: gs://arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX15016115.h5ad


## 5b. IO × disease-theme cohort cards (cells + SRX)

Repeat the same pattern for each **disease theme**: intersect the IO keyword slice with rows whose **`disease` or `tissue`** text matches a theme regex. Summarize **SRX count**, **cell count** (`obs_count` sum), and **distinct raw `disease` strings** (subtype richness in label space—not clinical staging).

Edit **`DISEASE_THEMES`** to add or narrow patterns (e.g. hematologic vs solid). Optional **`THEME_SUBTYPE_RULES`** buckets raw labels into a short list for card text (e.g. ER+/HER2+ style groupings when you define regexes yourself).

In [141]:
io_row_mask = (
    io_mask_series(sample["perturbation"])
    | io_mask_series(sample["disease"])
    | io_mask_series(sample["tissue"])
)

# Edit this map: each theme = regex over disease OR tissue text, intersected with IO slice above.
DISEASE_THEMES: dict[str, re.Pattern[str]] = {
    "breast": re.compile(
        r"breast|mammary|DCIS|ductal carcinoma|lobular|HER2|triple[- ]negative|ER\+|PR\+",
        re.I,
    ),
    "lung": re.compile(
        r"\blung\b|pulmonary|NSCLC|SCLC|adenocarcinoma of the lung|mesothelioma|pleura",
        re.I,
    ),
    "melanoma": re.compile(r"melanoma", re.I),
    "colorectal": re.compile(
        r"colorectal|colon cancer|rectal cancer|CRC\b",
        re.I,
    ),
    "hematologic": re.compile(
        r"leukemia|leukaemia|lymphoma|myeloma|myeloid|\bCML\b|\bAML\b|\bALL\b|\bCLL\b|DLBCL|multiple myeloma",
        re.I,
    ),
    "head_and_neck": re.compile(
        r"head and neck|HNSCC|oral cavity|oropharyn|laryngeal",
        re.I,
    ),
}


def disease_or_tissue_matches(pat: re.Pattern[str], frame: pd.DataFrame) -> pd.Series:
    d = frame["disease"].fillna("").astype(str)
    t = frame["tissue"].fillna("").astype(str)
    return d.str.contains(pat, na=False) | t.str.contains(pat, na=False)


def io_theme_cohort(
    sample_df: pd.DataFrame, io_mask: pd.Series, theme_pat: re.Pattern[str]
) -> pd.DataFrame:
    base = sample_df.loc[io_mask].copy()
    return base.loc[disease_or_tissue_matches(theme_pat, base)]


def format_io_theme_card(
    theme_name: str,
    cohort: pd.DataFrame,
    *,
    top_disease: int = 8,
    top_perturbation: int = 6,
    top_tissue: int = 5,
) -> str:
    if cohort.empty:
        return f"Theme: {theme_name}\n(no rows in IO ∩ theme slice)"

    n_srx = cohort["srx_accession"].nunique()
    w = pd.to_numeric(cohort["obs_count"], errors="coerce").fillna(0)
    n_cells = int(w.sum())
    n_labels = cohort["disease"].nunique()

    lines = [
        f"Theme: {theme_name}",
        f"In this IO-filtered slice we have {n_srx:,} SRX experiments and {n_cells:,} cells (obs_count sum).",
        f"Distinct raw disease strings: {n_labels:,} (label heterogeneity in metadata).",
        "",
        "Top disease labels by cell count:",
    ]
    g = cohort.groupby("disease", dropna=False)["obs_count"].sum().sort_values(ascending=False)
    for lab, cells in g.head(top_disease).items():
        lines.append(f"  - {lab!s}: {int(cells):,} cells")

    lines.append("")
    lines.append("Top perturbation labels by cell count:")
    g2 = cohort.groupby("perturbation", dropna=False)["obs_count"].sum().sort_values(ascending=False)
    for lab, cells in g2.head(top_perturbation).items():
        lines.append(f"  - {lab!s}: {int(cells):,} cells")

    lines.append("")
    lines.append("Top tissue labels by cell count:")
    g3 = cohort.groupby("tissue", dropna=False)["obs_count"].sum().sort_values(ascending=False)
    for lab, cells in g3.head(top_tissue).items():
        lines.append(f"  - {lab!s}: {int(cells):,} cells")

    return "\n".join(lines)


# Optional: bucket raw disease text into a short list for narrative cards (patterns are yours to refine).
THEME_SUBTYPE_RULES: dict[str, list[tuple[re.Pattern[str], str]]] = {
    "breast": [
        (re.compile(r"triple[- ]negative|TNBC", re.I), "triple_negative"),
        (re.compile(r"HER2|ERBB2", re.I), "HER2_mentioned"),
        (re.compile(r"ER\+|estrogen receptor positive|ER positive", re.I), "ER_positive_mentioned"),
        (re.compile(r"PR\+|progesterone receptor positive", re.I), "PR_positive_mentioned"),
    ],
}


def subtype_bucket_cell_counts(
    cohort: pd.DataFrame, rules: list[tuple[re.Pattern[str], str]]
) -> pd.Series:
    w = pd.to_numeric(cohort["obs_count"], errors="coerce").fillna(0)
    s = cohort["disease"].fillna("").astype(str)
    out: dict[str, int] = {}
    for pat, name in rules:
        out[name] = int(w[s.str.contains(pat, na=False)].sum())
    return pd.Series(out, dtype="int64")


cohorts: dict[str, pd.DataFrame] = {
    name: io_theme_cohort(sample, io_row_mask, pat) for name, pat in DISEASE_THEMES.items()
}

overview_rows = []
for theme_name, ch in cohorts.items():
    w = pd.to_numeric(ch["obs_count"], errors="coerce").fillna(0)
    overview_rows.append(
        {
            "theme": theme_name,
            "n_srx": ch["srx_accession"].nunique(),
            "n_cells": int(w.sum()),
            "n_distinct_disease": ch["disease"].nunique(),
        }
    )

theme_overview = pd.DataFrame(overview_rows).sort_values("n_cells", ascending=False)
theme_overview

,theme,n_srx,n_cells,n_distinct_disease
4,hematologic,506,4266537,97
5,head_and_neck,184,1050835,13
2,melanoma,175,978984,32
1,lung,98,896808,43
0,breast,40,201069,12
3,colorectal,1,4053,1


### Theme check against raw labels

**Coverage:** how many SRX rows match **0 / 1 / 2+** theme regexes (on `disease` **or** `tissue`)? Rows can match multiple themes if strings fit several patterns.

**Per theme:** SRX and cell totals on the **full** sample (same as above, **not** IO-filtered)—compare to `theme_overview`, which is **IO ∩ theme**.

**Top diseases inside each theme** helps you tighten or split regexes (e.g. move borderline strings into a new theme).

In [142]:
import numpy as np
from IPython.display import display

theme_match_matrix = np.column_stack(
    [disease_or_tissue_matches(pat, sample).to_numpy() for pat in DISEASE_THEMES.values()]
)
n_hit = theme_match_matrix.sum(axis=1)
hit_dist = pd.Series(n_hit).value_counts().sort_index()
hit_dist.index.name = "n_themes_matched"
hit_dist.name = "n_srx_rows"
print("How many theme patterns match each SRX row (disease OR tissue text):")
display(hit_dist.to_frame())

rows_full = []
for name, pat in DISEASE_THEMES.items():
    m = disease_or_tissue_matches(pat, sample)
    sub = sample.loc[m]
    w = pd.to_numeric(sub["obs_count"], errors="coerce").fillna(0)
    rows_full.append(
        {"theme": name, "n_srx": len(sub), "n_cells": int(w.sum())}
    )
theme_on_full_sample = pd.DataFrame(rows_full).sort_values("n_cells", ascending=False)
display(theme_on_full_sample)

TOP_DISEASES_PER_THEME = 12
for name, pat in DISEASE_THEMES.items():
    sub = sample.loc[disease_or_tissue_matches(pat, sample)]
    if sub.empty:
        print(f"\n--- theme {name!r}: (no rows) ---")
        continue
    g = (
        sub.groupby("disease", dropna=False)["obs_count"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_DISEASES_PER_THEME)
    )
    print(f"\n--- theme {name!r}: top diseases by cells ---")
    display(g.astype(int).to_frame("cells"))

How many theme patterns match each SRX row (disease OR tissue text):


,n_srx_rows
n_themes_matched,
0,26235
1,8980
2,50
4,1


,theme,n_srx,n_cells
4,hematologic,4693,55151934
1,lung,2300,14038920
0,breast,1134,8500620
2,melanoma,394,2196520
5,head_and_neck,305,1506408
3,colorectal,258,1444953



--- theme 'breast': top diseases by cells ---


,cells
disease,
breast cancer,1983288
unsure,752569
normal,719262
Hormone dependent breast cancer (HDBC),312471
triple-negative breast cancer,264462
hormone-dependent breast cancer,192351
hormone dependent breast cancer,164984
triple-negative breast cancer (TNBC),153518
Breast cancer,131719


...

,cells
disease,
colorectal cancer,637126
colorectal carcinoma,211801
colon cancer,63004
Colorectal carcinoma,47906
related to rectal cancer surgery outcomes,47623
colorectal liver metastasis,29348
"colorectal cancer, precancerous adenomas",26094
Colon cancer,23735
"breast neoplasms, colorectal neoplasms",18364



--- theme 'hematologic': top diseases by cells ---


,cells
disease,
chronic myelogenous leukemia,13421381
unsure,9600770
acute T cell leukemia,5550486
Chronic Myelogenous Leukemia,4336204
Chronic Myelogenous Leukemia (CML),1968445
multiple myeloma,1618395
Acute T cell leukemia,1505159
chronic myelogenous leukemia (CML),1101162
Chronic Myeloid Leukemia,999017



--- theme 'head_and_neck': top diseases by cells ---


,cells
disease,
Head and neck squamous cell carcinoma (HNSCC),457052
head and neck squamous cell carcinoma,262334
Head and neck squamous cell carcinoma,246636
head and neck squamous cell carcinoma (HNSCC),114110
HPV-positive head and neck squamous cell carcinoma,89420
Head and Neck Squamous Cell Carcinoma (HNSCC),31403
brain metastasis from laryngeal squamous cell carcinoma,30101
head and neck squamous cell carcinoma (HPV+ and HPV-),23753
HPV+ head and neck squamous cell carcinoma,17153


In [143]:
sample["srx_accession"].nunique()

35266

In [144]:
for theme in theme_overview.loc[theme_overview["n_cells"] > 0, "theme"]:
    print(format_io_theme_card(theme, cohorts[theme]))
    print("=" * 72)

for theme, rules in THEME_SUBTYPE_RULES.items():
    ch = cohorts[theme]
    if ch.empty:
        continue
    bc = subtype_bucket_cell_counts(ch, rules)
    if bc.sum() == 0:
        continue
    print(f"\nOptional disease-text buckets for theme {theme!r} (cells; rows can match multiple patterns):")
    print(bc[bc > 0].to_string())

Theme: hematologic
In this IO-filtered slice we have 506 SRX experiments and 4,266,537 cells (obs_count sum).
Distinct raw disease strings: 97 (label heterogeneity in metadata).

Top disease labels by cell count:
  - multiple myeloma: 1,154,158 cells
  - Multiple Myeloma, early relapse: 420,943 cells
  - multiple myeloma, early relapse: 397,065 cells
  - Multiple Myeloma: 329,530 cells
  - bone marrow failure after CD19 CAR-T for Richter transformed DLBCL: 146,884 cells
  - follicular lymphoma: 144,690 cells
  - Multiple Myeloma, Early relapse: 140,020 cells
  - bone marrow failure and DLBCL: 102,711 cells

Top perturbation labels by cell count:
  - BCMA CAR-T cell therapy: 1,945,017 cells
  - BCMA CAR-T therapy: 216,583 cells
  - CAR T therapy, IL-4 addition: 164,856 cells
  - CD19-directed CAR T-cell therapy (tisagenlecleucel): 146,884 cells
  - CD19 CAR-T therapy (tisagenlecleucel): 102,711 cells
  - RCHOP immuno-chemotherapy: 101,952 cells

Top tissue labels by cell count:
  - bone

## 6. Patient / stage / progression fields — coverage in these parquets

`sample_metadata` exposes study-level disease, tissue, perturbation, and technology fields. There is **no** dedicated patient ID or cancer **stage** column in the schema we loaded.

Optional: scan string fields for words like `stage`, `grade`, `TNM`, `metastatic` to see if they appear inside free-text labels.

In [145]:
STAGE_RE = re.compile(
    r"stage\s*[IV0-4]|TNM|metastatic|metastasis|grade\s*[0-4]|AJCC",
    re.I,
)

text_cols = [c for c in sample.columns if sample[c].dtype == object]
hits = []
for c in text_cols:
    m = sample[c].fillna("").astype(str).str.contains(STAGE_RE, na=False)
    if m.any():
        hits.append((c, int(m.sum())))
hits = sorted(hits, key=lambda x: -x[1])
print("Column -> rows with stage-like tokens:")
for c, k in hits[:15]:
    print(f"  {c}: {k}")
if not hits:
    print("  (none)")

Column -> rows with stage-like tokens:
  (none)


In [146]:
if hits:
    c = hits[0][0]
    m = sample[c].fillna("").astype(str).str.contains(STAGE_RE, na=False)
    sample.loc[m, c].value_counts().head(15)

## 7. Optional: cell types for a **small** SRX set via `obs_metadata`

The Homo sapiens `obs_metadata` parquet can be hundreds of millions of rows. The cell below uses a PyArrow **dataset filter** on `SRX_accession` so only matching rows are materialized (runtime still depends on how well row-group statistics prune the scan).

For heavier summaries, consider DuckDB `read_parquet(..., hive_partitioning=...)` or Polars lazy scans.

In [147]:
TOP_N = 5
top_srx = io.sort_values("obs_count", ascending=False).head(TOP_N)["srx_accession"].tolist()
top_srx

['SRX15016115', 'SRX22671373', 'SRX22671374', 'SRX15016116', 'SRX23973422']

In [148]:
import pyarrow as pa
import pyarrow.dataset as ds

use_cols = ["SRX_accession", "cell_type", "cell_ontology_term_id"]
if not top_srx:
    print("No IO rows selected; widen IO keywords or check data.")
else:
    dset = ds.dataset(str(OBS_PARQUET), format="parquet")
    tbl = dset.to_table(
        columns=use_cols,
        filter=ds.field("SRX_accession").isin(pa.array(top_srx)),
    )
    obs_sub = tbl.to_pandas()
    print(f"Loaded {len(obs_sub):,} cell rows across {len(top_srx)} SRX accessions")
    (
        obs_sub.groupby(["SRX_accession", "cell_type"], dropna=False)
        .size()
        .sort_values(ascending=False)
        .head(35)
    )

Loaded 583,840 cell rows across 5 SRX accessions
